# 3. Data Preparation: Pembersihan Data, Reduksi Fitur Redundan, dan Rekayasa 13 Fitur Spektral Pasti

Tahap **Data Preparation** dalam metodologi **CRISP-DM** bertugas mentransformasikan data mentah hasil ekstraksi reflektansi Sentinel-2 menjadi representasi fitur bernilai tinggi yang siap digunakan oleh algoritma pembelajaran mesin.

### Pembaruan & Penyederhanaan Terpadu (Clean & Robust Pipeline):
1. **Pembersihan Data (*Data Cleaning & Quality Assurance*)**:
   - Eliminasi piksel jenuh awan (*cloud saturation* > 1.0/0.95).
   - Pembersihan sampel anomali kontradiktif (misal: perairan dengan NDVI tinggi/vegetatif, atau poligon bangunan yang sebenarnya kebun hijau pekat).
2. **Reduksi Fitur (*Feature Simplification*)**:
   - Memangkas indeks-indeks redundan, collinear, dan noisy (seperti EBBI, UI, IBI, VR, SAVI, MSAVI, TC_Wetness).
   - Mempertahankan hanya **13 Fitur Pasti dan Teruji Secara Fisika**:
     - **7 Band Reflektansi Asli BOA**: `B02` (Blue), `B03` (Green), `B04` (Red), `B08` (NIR Broad), `B8A` (NIR Narrow), `B11` (SWIR-1), `B12` (SWIR-2).
     - **6 Indeks Biofisik Terpilih**: `NDVI`, `MNDWI`, `NDBI`, `Mangrove_Index_8A_11_4`, `EVI`, `BSI`.
3. **Penyelarasan Skala & Pembagian Data (Stratified 80:20 Split)**:
   - Mempertahankan distribusi proporsional 5 kelas tutupan lahan pada data latih (*training*) dan data uji (*testing*).


In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('[OK] Seluruh library Data Preparation berhasil dimuat.')

[OK] Seluruh library Data Preparation berhasil dimuat.


## 3.1 Pemuatan Data dan Pembersihan Kualitas Sampel Poligon (*Polygon Cleaning*)

Data mentah poligon dimuat dari direktori `2-data_understanding/extracted_data/sentinel2_sample_polygons.csv`. Sebelum rekayasa fitur, dilakukan pembersihan terhadap anomali spektral dan nilai reflektansi jenuh.


In [2]:
data_dir = os.path.join('..', '2-data_understanding', 'extracted_data')
raw_path = os.path.join(data_dir, 'sentinel2_sample_polygons.csv')
df_raw = pd.read_csv(raw_path)

print(f'Dimensi Dataset Poligon Mentah: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom')
print(f'Total Nilai Hilang (NaN): {df_raw.isnull().sum().sum()}')
print(f'Total Nilai Tak Terhingga (Inf): {np.isinf(df_raw.select_dtypes(include=np.number)).values.sum()}')
print(f'Total Data Duplikat: {df_raw.duplicated().sum()}')

# Prosedur Pembersihan Data Poligon Fisis
def clean_poly_func(r):
    bands = [r['B02'], r['B03'], r['B04'], r['B08'], r['B8A'], r['B11'], r['B12']]
    if any(b <= 0.0 or b >= 1.0 for b in bands): return False
    c = r['class_name']
    ndvi = r['NDVI']
    if c == 'perairan' and (ndvi > 0.15 or r['B08'] > 0.16): return False
    if c == 'bangunan' and (ndvi > 0.32 and r['NDBI'] < 0.05): return False
    if c == 'hutan' and ndvi < 0.35: return False
    if c == 'mangrove' and ndvi < 0.38: return False
    return True

df = df_raw[df_raw.apply(clean_poly_func, axis=1)].copy().reset_index(drop=True)
class_map = {'bangunan': 0, 'pertanian': 1, 'perairan': 2, 'hutan': 3, 'mangrove': 4}
df['label'] = df['class_name'].map(class_map)

print(f'\nDimensi Setelah Pembersihan: {df.shape[0]} poligon valid')
print('\nDistribusi Poligon Bersih per Kelas Tutupan Lahan:')
display(df['class_name'].value_counts().reset_index())


Dimensi Dataset Poligon Mentah: 250 baris, 18 kolom
Total Nilai Hilang (NaN): 0
Total Nilai Tak Terhingga (Inf): 0
Total Data Duplikat: 0

Dimensi Setelah Pembersihan: 217 poligon valid

Distribusi Poligon Bersih per Kelas Tutupan Lahan:


,class_name,count
0,pertanian,50
1,bangunan,47
2,perairan,43
3,mangrove,42
4,hutan,35


## 3.2 Rekayasa 13 Fitur Spektral Pasti (*Feature Engineering & Simplification*)

Mereduksi fitur menjadi **6 Indeks Biofisik Terpilih** yang memiliki dasar fisis jelas:
1. **`NDVI` (*Normalized Difference Vegetation Index*)**: Mengukur kerapatan klorofil kanopi vegetasi umum.
2. **`MNDWI` (*Modified Normalized Difference Water Index*)**: Mendeteksi badan air permukaan dan kebasahan substrat tanpa tertukar dengan bayangan perkotaan.
3. **`NDBI` (*Normalized Difference Built-up Index*)**: Menangkap pantulan tinggi material atap/beton/aspal pada pita SWIR-1 dibanding NIR.
4. **`Mangrove_Index_8A_11_4`**: Rasio khusus Sentinel-2 untuk memisahkan vegetasi mangrove pesisir berlumpur payau dari hutan daratan kering.
5. **`EVI` (*Enhanced Vegetation Index*)**: Mengestimasi biomassa kanopi lebat tanpa mengalami kejenuhan (*saturation*) optik seperti NDVI.
6. **`BSI` (*Bare Soil Index*)**: Memisahkan lahan terbuka/tanah gersang pertanian pra-tanam dari material buatan manusia.


In [3]:
eps = 1e-6
b02, b03, b04 = df['B02'].values, df['B03'].values, df['B04'].values
b08, b8a = df['B08'].values, df['B8A'].values
b11, b12 = df['B11'].values, df['B12'].values

# 6 Indeks Biofisik Terpilih & Pasti
df['NDVI'] = (b08 - b04) / (b08 + b04 + eps)
df['MNDWI'] = (b03 - b11) / (b03 + b11 + eps)
df['NDBI'] = (b11 - b08) / (b11 + b08 + eps)
df['Mangrove_Index_8A_11_4'] = (b8a - b04) / (b11 + eps)
df['EVI'] = 2.5 * (b08 - b04) / (b08 + 6.0 * b04 - 7.5 * b02 + 1.0 + eps)
df['BSI'] = ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + eps)

# 13 Fitur Prediktor Inti (7 Band + 6 Indeks)
feature_cols = [
    'B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12',
    'NDVI', 'MNDWI', 'NDBI', 'Mangrove_Index_8A_11_4', 'EVI', 'BSI'
]

print(f'[OK] Berhasil merekayasa {len(feature_cols)} Fitur Inti (7 Band Reflektansi + 6 Indeks Biofisik Teruji):')
for i, fcol in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {fcol}')

# Ringkasan Statistik 13 Fitur per Kelas
display(df.groupby('class_name')[feature_cols].mean().round(4))


[OK] Berhasil merekayasa 13 Fitur Inti (7 Band Reflektansi + 6 Indeks Biofisik Teruji):
   1. B02
   2. B03
   3. B04
   4. B08
   5. B8A
   6. B11
   7. B12
   8. NDVI
   9. MNDWI
  10. NDBI
  11. Mangrove_Index_8A_11_4
  12. EVI
  13. BSI


,B02,B03,B04,B08,B8A,B11,B12,NDVI,MNDWI,NDBI,Mangrove_Index_8A_11_4,EVI,BSI
class_name,,,,,,,,,,,,,
bangunan,0.1709,0.1887,0.2123,0.2473,0.2569,0.3116,0.2921,0.0991,-0.2853,0.1192,0.1724,0.0643,0.1258
hutan,0.0472,0.0673,0.0567,0.2724,0.2884,0.1842,0.1075,0.6719,-0.4717,-0.2055,1.3348,0.4283,-0.1547
mangrove,0.0380,0.0640,0.0409,0.2946,0.3081,0.1337,0.0632,0.7576,-0.3349,-0.3887,2.1875,0.5037,-0.3259
perairan,0.1028,0.1091,0.0926,0.0776,0.0826,0.0720,0.0589,-0.0827,0.2210,-0.0617,-0.3773,-0.0485,-0.0627
pertanian,0.0770,0.1036,0.1132,0.2436,0.2598,0.2339,0.1658,0.3802,-0.3692,-0.0410,0.7156,0.2476,0.0147


## 3.3 Pembagian Data (Stratified 80:20 Split) dan Penskalaan Fitur

Untuk memastikan evaluasi yang obyektif, dataset poligon dibagi dengan rasio 80:20 menggunakan stratifikasi kelas target.


In [4]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=2024, stratify=df['label'])

prep_dir = 'prepared_data'
os.makedirs(prep_dir, exist_ok=True)

# Simpan Dataset Poligon Bersih
df.to_csv(os.path.join(prep_dir, 'data_prepared_all.csv'), index=False)
train_df.to_csv(os.path.join(prep_dir, 'train_data.csv'), index=False)
test_df.to_csv(os.path.join(prep_dir, 'test_data.csv'), index=False)

# Sinkronisasi ke direktori deployment
deploy_prep_dir = os.path.join('..', '6-deployment', 'prepared_data')
if os.path.exists(deploy_prep_dir):
    df.to_csv(os.path.join(deploy_prep_dir, 'data_prepared_all.csv'), index=False)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(train_df[feature_cols])
X_te_scaled = scaler.transform(test_df[feature_cols])
joblib.dump(scaler, os.path.join(prep_dir, 'scaler.pkl'))

print(f'[OK] Dataset Poligon Berhasil Disimpan:')
print(f'  - Total Data Poligon : {len(df)} sampel')
print(f'  - Data Poligon Latih : {len(train_df)} sampel')
print(f'  - Data Poligon Uji   : {len(test_df)} sampel')


[OK] Dataset Poligon Berhasil Disimpan:
  - Total Data Poligon : 217 sampel
  - Data Poligon Latih : 173 sampel
  - Data Poligon Uji   : 44 sampel


## 3.4 Pembersihan dan Rekayasa Dataset Spektral Piksel (*Pixel-Level Dataset Preparation*)

Menerapkan filter pembersihan fisis ketat pada dataset piksel:
- Menghapus piksel awan jenuh (> 1.0).
- Menghapus piksel perairan yang terkontaminasi vegetasi liar (NDVI > 0.12 atau NIR > 0.14).
- Menghapus piksel bangunan yang keliru tersampel ke kebun hijau lebat (NDVI > 0.30 dan NDBI < 0.02).


In [5]:
pixels_raw_path = os.path.join(data_dir, 'sentinel2_pixels_dataset.csv')
df_pixels_raw = pd.read_csv(pixels_raw_path)
print(f'Dimensi Dataset Piksel Awal: {df_pixels_raw.shape[0]} baris, {df_pixels_raw.shape[1]} kolom')

# Filter Pembersihan Piksel Fisis
def clean_pixel(r):
    bands = [r['B02'], r['B03'], r['B04'], r['B08'], r['B8A'], r['B11'], r['B12']]
    if any(b <= 0.0 or b >= 1.0 for b in bands):
        return False
    c = r['class_name']
    ndvi = r['NDVI']
    b8 = r['B08']
    if c == 'perairan':
        if ndvi > 0.12 or b8 > 0.14: return False
    elif c == 'bangunan':
        if ndvi > 0.30 and r['NDBI'] < 0.02: return False
    elif c == 'hutan':
        if ndvi < 0.40: return False
    elif c == 'mangrove':
        if ndvi < 0.40: return False
    return True

clean_pixels = df_pixels_raw[df_pixels_raw.apply(clean_pixel, axis=1)].copy().reset_index(drop=True)

# Hitung 6 Indeks Biofisik Terpilih
eps = 1e-6
pb02, pb03, pb04 = clean_pixels['B02'].values, clean_pixels['B03'].values, clean_pixels['B04'].values
pb08, pb8a = clean_pixels['B08'].values, clean_pixels['B8A'].values
pb11, pb12 = clean_pixels['B11'].values, clean_pixels['B12'].values

clean_pixels['NDVI'] = (pb08 - pb04) / (pb08 + pb04 + eps)
clean_pixels['MNDWI'] = (pb03 - pb11) / (pb03 + pb11 + eps)
clean_pixels['NDBI'] = (pb11 - pb08) / (pb11 + pb08 + eps)
clean_pixels['Mangrove_Index_8A_11_4'] = (pb8a - pb04) / (pb11 + eps)
clean_pixels['EVI'] = 2.5 * (pb08 - pb04) / (pb08 + 6.0 * pb04 - 7.5 * pb02 + 1.0 + eps)
clean_pixels['BSI'] = ((pb11 + pb04) - (pb08 + pb02)) / ((pb11 + pb04) + (pb08 + pb02) + eps)
clean_pixels['label'] = clean_pixels['class_name'].map(class_map)

print(f'Dimensi Dataset Piksel Setelah Pembersihan: {clean_pixels.shape[0]} piksel valid')

# Split Piksel Stratified 80:20
train_pix, test_pix = train_test_split(clean_pixels, test_size=0.20, random_state=2024, stratify=clean_pixels['label'])

clean_pixels.to_csv(os.path.join(prep_dir, 'pixels_prepared_all.csv'), index=False)
train_pix.to_csv(os.path.join(prep_dir, 'train_pixels.csv'), index=False)
test_pix.to_csv(os.path.join(prep_dir, 'test_pixels.csv'), index=False)

# Sinkronisasi ke direktori deployment
if os.path.exists(deploy_prep_dir):
    clean_pixels.to_csv(os.path.join(deploy_prep_dir, 'pixels_prepared_all.csv'), index=False)
    train_pix.to_csv(os.path.join(deploy_prep_dir, 'train_pixels.csv'), index=False)
    test_pix.to_csv(os.path.join(deploy_prep_dir, 'test_pixels.csv'), index=False)

print(f'[OK] Dataset Piksel Berhasil Disimpan:')
print(f'  - Total Piksel Bersih : {len(clean_pixels)} sampel')
print(f'  - Data Piksel Latih   : {len(train_pix)} sampel')
print(f'  - Data Piksel Uji     : {len(test_pix)} sampel')
print('\nDistribusi Piksel per Kelas Tutupan Lahan:')
display(clean_pixels['class_name'].value_counts().reset_index())


Dimensi Dataset Piksel Awal: 12500 baris, 15 kolom
Dimensi Dataset Piksel Setelah Pembersihan: 10469 piksel valid
[OK] Dataset Piksel Berhasil Disimpan:
  - Total Piksel Bersih : 10469 sampel
  - Data Piksel Latih   : 8375 sampel
  - Data Piksel Uji     : 2094 sampel

Distribusi Piksel per Kelas Tutupan Lahan:


,class_name,count
0,pertanian,2500
1,bangunan,2303
2,mangrove,2156
3,perairan,1886
4,hutan,1624
